# QFT-Graph: A-4 U(1) variant experiments (Colab GPU runner)

Runs the committed CLI script `scripts/train_u1.py` on the labeled U(1)
ensembles in `data/u1_configs/` (plan ground rule 4: this notebook is only a
launcher). Every run writes `results/<run_id>.json` + a checkpoint under
`experiments/runs/u1/` (reused by A-5). The driver **skips runs whose
results JSON already exists**, so after a disconnect just re-run the cell —
at most the in-progress run is lost.

The matrix reflects the **pilot-informed restructuring**
(results/u1pilot_*.json, decision-record status notes, 2026-07-11):
at the frozen protocol Variants A/B sit at chance on all gauge-invariant
targets (no first-order signal in raw links — gauge symmetry), while C is
near ceiling with receptive-field degradation at large loops. So: establish
the A/B null robustly + escape hatches (E1, E2), run the full protocol on C
(E3), and measure the receptive-field heatmap on C (E4). Sections are
independent — run in any order, split across sessions freely.

| Cell | Experiment | Runs | est. GPU time* |
|---|---|---|---|
| E1 | A/B null robustness: 3 (beta, L) points x 2 variants x 3 seeds | 18 | ~2-4 h |
| E2 | Escape hatches (A @ L8 beta2): 450 ep / low LR / W1x1-only / B=6 | 4 | ~1-2 h |
| E3 | C full protocol: beta {0.5,1,2,3,4} x L {8,16} x 5 seeds | 50 | ~4-8 h |
| E3b | (optional) C at L=32: beta {1,2,4} x 3 seeds | 9 | ~3-6 h |
| E4 | Receptive field on C: B {2,3,4,6} x (L16, beta {2,4}) x 3 seeds | 24 | ~2-4 h |
| E5 | (optional) parameter-matched arm @ (L8, beta2) x 3 seeds | 9 | ~1-2 h |

*Laptop-CPU reference: 150 epochs at L=8 took 54 min (A) / 21 min (B, C).
These graphs are small, so the DataLoader can bottleneck before the GPU
does — treat estimates as rough and watch the first run's timing.

In [ ]:
import os, sys

IN_COLAB = 'COLAB_GPU' in os.environ or 'COLAB_RELEASE_TAG' in os.environ
assert IN_COLAB, 'This runner is meant for Colab (use scripts/train_u1.py locally)'

from google.colab import drive
drive.mount('/content/drive')
PROJECT_ROOT = '/content/drive/MyDrive/qft_graph'
os.chdir(PROJECT_ROOT)

!pip install -q torch-geometric omegaconf h5py

# Make qft_graph importable for the CLI invocations below
os.environ['PYTHONPATH'] = os.path.join(PROJECT_ROOT, 'src')

import torch
print('CUDA available:', torch.cuda.is_available(),
      '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## Physics gate (ground rule 1)

No training run launches until the exact-value tests pass **in this
environment** — Colab's torch/PyG versions differ from the laptop's.
~2-5 min. If anything fails, stop and report; do not weaken tests.

In [ ]:
!python -m pytest tests/ -q

## Run driver

Builds CLI commands for `scripts/train_u1.py`, skips completed runs
(matching the script's `run_id` format exactly), streams output, and
continues past failures. Use full variant names — they appear in filenames.

In [ ]:
import subprocess, sys
from pathlib import Path

DATA = lambda L, beta: f'data/u1_configs/u1_L{L}_beta{beta:g}.h5'

def run_id(r):
    # Must mirror scripts/train_u1.py exactly (prefix_variant_stem_H_B_seed)
    stem = Path(r['data']).stem
    return (f"{r['prefix']}_{r['variant']}_{stem}"
            f"_H{r.get('H', 64)}_B{r.get('B', 3)}_seed{r['seed']}")

def launch(runs):
    failures = []
    for i, r in enumerate(runs, 1):
        rid = run_id(r)
        if (Path('results') / f'{rid}.json').exists():
            print(f'[{i}/{len(runs)}] SKIP (done): {rid}')
            continue
        # Argument list (no shell): immune to spaces in paths, portable
        args = [sys.executable, 'scripts/train_u1.py',
                '--data', r['data'],
                '--variant', r['variant'],
                '--seeds', str(r['seed']),
                '--hidden_dim', str(r.get('H', 64)),
                '--n_mp_blocks', str(r.get('B', 3)),
                '--epochs', str(r.get('epochs', 150)),
                '--lr', str(r.get('lr', 1e-3)),
                '--run_prefix', r['prefix'],
                *r.get('extra', '').split()]
        print(f'[{i}/{len(runs)}] RUN: {rid}', flush=True)
        rc = subprocess.run(args).returncode
        if rc != 0:
            failures.append(rid)
            print(f'  FAILED rc={rc}')
    print(f'\n{len(runs) - len(failures)}/{len(runs)} ok')
    if failures:
        print('failed:', failures)

## E1 — A/B null robustness (3 seeds x 3 ensemble points)

The pilot null (r ~ 0 on all targets, L=8 beta=2 seed 0) needs to be shown
robust across seeds, beta, and volume before it can carry weight in the
paper. beta=1 adds a disordered-regime point; L=16 checks volume.

In [ ]:
runs = [
    dict(prefix='a4null', variant=v, data=DATA(L, beta), seed=s)
    for (L, beta) in [(8, 2.0), (8, 1.0), (16, 2.0)]
    for v in ('link_nodes', 'edge_features')
    for s in (0, 1, 2)
]
launch(runs)

## E2 — Escape hatches (Variant A, L=8 beta=2, seed 0)

Referee-proofing the null: does ANY cheap change lift A off chance?
(a) 3x epochs; (b) low LR, 2x epochs; (c) single-task on the most local
target (W1x1 only, action/Q loss weights zeroed); (d) deeper receptive
field (B=6). If all four stay at r ~ 0, the null is well-established at
this scale; anything that lifts off is a lead worth chasing.

In [ ]:
runs = [
    dict(prefix='a4hatch450ep', variant='link_nodes', data=DATA(8, 2.0), seed=0,
         epochs=450),
    dict(prefix='a4hatchlowlr', variant='link_nodes', data=DATA(8, 2.0), seed=0,
         epochs=300, lr=3e-4),
    dict(prefix='a4hatchw11', variant='link_nodes', data=DATA(8, 2.0), seed=0,
         epochs=300, extra='--wilson_loops 1x1 --w_action 0 --w_q 0'),
    dict(prefix='a4hatchdeep', variant='link_nodes', data=DATA(8, 2.0), seed=0,
         B=6),
]
launch(runs)

## E3 — Variant C full protocol (the ceiling numbers)

5 seeds x beta {0.5, 1, 2, 3, 4} x L {8, 16}. These are the headline
C rows of the A/B/C table and the anchor for A-5's eps_gauge ceiling.
50 runs — safe to split across sessions (skip logic resumes).

In [ ]:
runs = [
    dict(prefix='a4C', variant='invariant_oracle', data=DATA(L, beta), seed=s)
    for L in (8, 16)
    for beta in (0.5, 1.0, 2.0, 3.0, 4.0)
    for s in range(5)
]
launch(runs)

## E3b — (optional) Variant C at L=32

Volume-scaling check at 3 beta points x 3 seeds. Heavier per run
(4x the sites of L=16); run only if session time allows.

In [ ]:
runs = [
    dict(prefix='a4C', variant='invariant_oracle', data=DATA(32, beta), seed=s)
    for beta in (1.0, 2.0, 4.0)
    for s in (0, 1, 2)
]
launch(runs)

## E4 — Receptive-field study on C (the heatmap)

B in {2, 3, 4, 6} blocks x loop areas {1, 4, 8, 9, 16} -> heatmap of
prediction r vs (area, B), per plan section 3 Task A-4. Run at L=16 for
two beta points: beta=2 (pilot point) and beta=4 (largest-loop labels
least noise-dominated — frac(W<=0) = 0.13 at 4x4 vs 0.47 at beta=2).
Degradation once loop diameter exceeds ~B lattice spacings is the
expected finding, not a bug — the pilot already showed 4x4 r = 0.47 at
B=3, beta=2, L=8.

In [ ]:
runs = [
    dict(prefix='a4rf', variant='invariant_oracle', data=DATA(16, beta), seed=s, B=B)
    for beta in (2.0, 4.0)
    for B in (2, 3, 4, 6)
    for s in (0, 1, 2)
]
launch(runs)

## E5 — (optional) parameter-matched arm (decision record 4)

With the A/B null, the matched arm mostly matters as due diligence: does
giving B/C the A@64 budget (H=102), or shrinking A to the B@64 budget
(H=40), change anything? Uses --match_params_to for provenance; the
skip-check H values are precomputed to mirror the resolved run_id.

In [ ]:
# Resolved matched-H values (verified: matched_hidden_dim at protocol point)
runs = []
for v, H, spec in [
    ('edge_features', 102, 'link_nodes:64'),
    ('invariant_oracle', 102, 'link_nodes:64'),
    ('link_nodes', 40, 'edge_features:64'),
]:
    for s in (0, 1, 2):
        runs.append(dict(
            prefix='a4match', variant=v, data=DATA(8, 2.0), seed=s, H=H,
            extra=f'--match_params_to {spec}',
        ))
launch(runs)

## Aggregate: seed mean +/- std per experiment cell

Quick look only — the committed table/figure scripts (next A-4 step on
the laptop) are the source of paper numbers. Do not edit JSONs by hand.

In [ ]:
import json, re
from collections import defaultdict
from pathlib import Path
import numpy as np

groups = defaultdict(list)
for p in sorted(Path('results').glob('a4*.json')):
    d = json.loads(p.read_text())
    gid = re.sub(r'_seed\d+$', '', d['run_id'])
    groups[gid].append(d['metrics']['test'])

KEYS = ['action_r', 'wilson_1x1_r', 'wilson_2x2_r', 'wilson_4x4_r', 'q_acc']
for gid in sorted(groups):
    ms = groups[gid]
    cols = '  '.join(
        f'{k}={np.mean([m[k] for m in ms]):+.3f}\u00b1{np.std([m[k] for m in ms]):.3f}'
        for k in KEYS if k in ms[0]
    )
    print(f'{gid}  (n={len(ms)})\n    {cols}')

## Done

Sync check: confirm `results/a4*.json` and `experiments/runs/u1/` show up
on the laptop's Drive mirror. Claude Code picks them up for the A/B/C
comparison table + receptive-field heatmap scripts, and A-5 reuses the
checkpoints for the gauge-copy eps_gauge evaluation.